I've come to find that the spotipy library is a helpful addon for accessing Spotify's API capabilities in this project.

In [9]:
import spotipy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
from spotipy.oauth2 import SpotifyClientCredentials
import time
import os
from dotenv import load_dotenv
import numpy as np
from scipy.io import wavfile
from scipy.fft import rfft, rfftfreq

API Credentials

I did some external research and found that for security reasons, I should try the python-dotenv package.

In [2]:
load_dotenv()

CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET))

data = []

Spotify imposed some changes within their API leading to some frustrations within the data collection process.
I've created a new loop to work with the data available to us.

I'm currently looking to supplement this data with another API to find song key + bpm.

In [3]:
with open('billboard_songs.txt', 'r', encoding='utf-8') as file:
    for line in file:
        if ',' not in line: 
            continue
            
        title, artist = line.strip().split(', ', 1)
        
        primary_artist = artist.split(' & ')[0].split(' Featuring ')[0]
        
        search = sp.search(q=f"track:{title} artist:{primary_artist}", type='track', limit=1)
        
        if search['tracks']['items']:
            track = search['tracks']['items'][0]
            
            row_data = {
                'Title': title, 
                'Artist': artist, 
                'Popularity': track['popularity'],
                'Duration_MS': track['duration_ms'],
                'Explicit': track['explicit'],
                'Release_Date': track['album']['release_date'],
                'Album_Name': track['album']['name']
            }
            
            data.append(row_data)  

        time.sleep(0.2)

Export the data to the spotify.csv

In [4]:
pd.DataFrame(data).to_csv('spotify.csv', index=False)

Merge metrics and spotify csv to data.csv

In [5]:
df1 = pd.read_csv('metrics.csv')
df2 = pd.read_csv('spotify.csv')

df= pd.merge(df1,df2,on=['Title','Artist'],how='inner')

pd.DataFrame(df).to_csv('data.csv', index=False)

In [ ]:
df = pd.read_csv('data.csv')
df['Hit'] = (df['Popularity'] >= 90).astype(int)
df = pd.get_dummies(df, columns=['Key'], drop_first=True)

In [18]:
def get_scipy_fourier(title):
    path = f"Tracks/WAV/{title}.wav"
    if not os.path.exists(path): 
        return pd.Series([np.nan, np.nan])
    
    try:
        sr, data = wavfile.read(path)
        
        # converting tracks to mono
        if data.ndim > 1: data = data.mean(axis=1)
        
        mid = len(data) // 2
        snippet = data[max(0, mid - 15*sr) : min(len(data), mid + 15*sr)]
        
        yf, xf = np.abs(rfft(snippet)), rfftfreq(len(snippet), 1/sr)
        
        return pd.Series([xf[np.argmax(yf)], np.sum(xf * yf) / (np.sum(yf) + 1e-9)])
    except:
        return pd.Series([np.nan, np.nan])

df[['Peak_Freq', 'Mean_Freq']] = df['Title'].apply(get_scipy_fourier)
print(df[['Peak_Freq', 'Mean_Freq']].head())

   Peak_Freq    Mean_Freq
0  54.633333  3443.499464
1  69.433333  3096.067638
2  49.000000  2903.022875
3  82.400000  2611.770990
4  88.666667  3835.166585


In [13]:
features = ['Duration_MS', 'BPM', 'Peak_Freq', 'Mean_Freq'] + [col for col in df.columns if 'Key_' in col]
df = df.dropna(subset=features + ['Hit'])

X = df[features]
y = df['Hit']

In [14]:
scaler = StandardScaler()
X_scale = scaler.fit_transform(X)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_scale, y)

y_pred = log_reg.predict(X_scale)

In [ ]:
print(confusion_matrix(y, y_pred))
print(classification_report(y, y_pred))
weights = pd.DataFrame({'Feature': X.columns, 'Weight': log_reg.coef_[0]})
print(weights.sort_values(by='Weight', key=abs, ascending=False))

[[3 0]
 [0 3]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

       Feature    Weight
3    Mean_Freq -0.777690
5      Key_11B -0.491225
0  Duration_MS  0.427907
8       Key_6B -0.402216
7       Key_5A -0.361944
1          BPM -0.344220
4      Key_11A  0.330294
2    Peak_Freq  0.113819
6       Key_3B  0.000000
